# Review and label SpecMine repositories

Load a random repository from the curated sample and display each spec's `spec_id` and markdown. After reviewing the specs, add the chosen IDs and labels to `examples/synthetic_labels.csv` using `spec_id,label`. Use `1` if AI was used at any point, `0` if it was not, and leave the label blank if the evidence is ambiguous. The rows already in that file are fabricated smoke-test labels; replace them before using it as real training data.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "specmine").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))
print(f"Project root: {project_root}")

from specmine.data import load_data

In [ ]:
SEED = 42
CURATED_SPEC_SAMPLE_SIZE = 500
LABELS_CSV = project_root / "examples" / "synthetic_labels.csv"

specs = load_data(
    curated_only=True,
    sample_size=CURATED_SPEC_SAMPLE_SIZE,
    seed=SEED,
)
if specs is None:
    raise RuntimeError("SpecMine data loading failed; no specs are available to review.")
if specs.empty:
    raise ValueError("The curated SpecMine sample contains no downloadable specs.")
required_columns = {"repo_name", "file_url_sha16", "local_path"}
missing_columns = required_columns - set(specs.columns)
if missing_columns:
    raise KeyError(f"Curated spec data is missing columns: {sorted(missing_columns)}")

repositories = specs["repo_name"].dropna().drop_duplicates()
if repositories.empty:
    raise ValueError("The curated sample contains no repository names.")
random_repo = repositories.sample(n=1).iloc[0]
repo_specs = specs.loc[specs["repo_name"].eq(random_repo)].sort_values("file_url_sha16")

print(f"Random repository: {random_repo}")
print(f"Specs to review: {len(repo_specs)}")
print(f"Labels CSV to update: {LABELS_CSV}")

In [ ]:
for row in repo_specs.itertuples(index=False):
    spec_id = str(row.file_url_sha16)
    spec_path = Path(row.local_path)
    if not spec_path.is_file():
        raise FileNotFoundError(f"Downloaded spec file is missing: {spec_path}")
    spec_markdown = spec_path.read_text(encoding="utf-8")
    print(f"Repository: {row.repo_name}")
    print(f"spec_id: {spec_id}")
    print(f"Source file: {spec_path}")
    display(Markdown(spec_markdown))
    display(Markdown("---"))

print(
    "For each reviewed spec, add one row to the CSV with its exact spec_id "
    "and label (0, 1, or blank). Do not duplicate an existing spec_id."
)